# Introduction to Django: Building a Point of Sale (POS) System 🛒

## Table of Contents
1. [What is Django?](#what-is-django)
2. [Why Use Django?](#why-use-django)
3. [Setting Up Your Django Environment](#setting-up-your-django-environment)
4. [Creating a Django Project](#creating-a-django-project)
5. [Understanding Django's Project Structure](#understanding-djangos-project-structure)
6. [Creating a Django App](#creating-a-django-app)
7. [Models and the Django ORM](#models-and-the-django-orm)
8. [Views and Templates](#views-and-templates)
9. [URL Routing in Django](#url-routing-in-django)
10. [Using Django Admin](#using-django-admin)
11. [Bonus: Django Shell — Interact with Your Data](#bonus-django-shell)
12. [Conclusion](#conclusion)
13. [Homework](#homework)

> **Project Goal**: By the end of this lesson, you will have the foundation to build a **Point of Sale (POS) System** where staff can browse products, process sales orders, and manage inventory through the admin panel.

---

## What is Django?

Django is a high-level Python web framework that encourages rapid development and clean, pragmatic design. Think of it as a **toolkit** — instead of building every part of a website from scratch, Django gives you ready-made pieces you can plug together.

**Real-world analogy**: Building a POS system without Django is like wiring up a cash register from scratch using raw electronic components. Django gives you the screen, the keypad, and the receipt printer already assembled — you just configure them for your store.

Key features of Django include:
- **Rapid Development**: Built-in tools for authentication, database management, and admin panels.
- **Scalability**: Used by **Instagram**, **Pinterest**, and **Disqus** to handle millions of users.
- **Security**: Automatically protects against SQL injection, XSS, and CSRF attacks.
- **Versatility**: Suitable for anything from a simple corner-shop till to a large retail chain.

---

## Why Use Django?

| Feature | Benefit |
|---|---|
| **Batteries Included** | Admin panel, authentication, ORM — all built-in |
| **MVT Architecture** | Clean separation of data (Model), logic (View), and design (Template) |
| **DRY Principle** | "Don't Repeat Yourself" — write code once, reuse everywhere |
| **Great Documentation** | One of the best-documented frameworks in Python |
| **Active Community** | Huge ecosystem of plugins and packages on [PyPI](https://pypi.org/) |

### Django's MVT Pattern Explained

```
User Request → URL Router → View (logic)
                                 ↓          ↑
                              Model      Template
                           (database)    (HTML)
```

For our POS system:
- **Model** → `Product`, `Order`, `OrderItem` tables in the database
- **View** → functions that fetch products/orders and pass them to templates
- **Template** → the HTML pages staff see on the screen

---

## Setting Up Your Django Environment

### Step 1: Install Python
Ensure Python 3.10+ is installed. Check your version:

```bash
python --version
```

### Step 2: Create a Virtual Environment (Recommended)
A virtual environment keeps your project's packages isolated from other Python projects.

In [ ]:
# Create a virtual environment
python -m venv .venv

# Activate it:
# On macOS/Linux:
source .venv/bin/activate

# On Windows:
.venv\Scripts\activate

# Your prompt will change to show (.venv) when active

### Step 3: Install Django

With your virtual environment active, install Django using `pip`:

In [ ]:
pip install django

Verify the installation:

In [ ]:
django-admin --version
# Expected output: 6.x.x

---

## Creating a Django Project

Let's create our POS System project. Run this in your terminal:

In [ ]:
# Create the project
django-admin startproject posdb

# Navigate into the project folder
cd posdb

---

## Understanding Django's Project Structure

This is what Django generates automatically:

posdb/                    ← root folder (your project)
    manage.py             ← command-line utility (run server, migrations, etc.)
    posdb/                ← the project "configuration package"
        __init__.py
        settings.py       ← all project settings (database, apps, time zone…)
        urls.py           ← top-level URL dispatcher
        asgi.py           ← async server entry point
        wsgi.py           ← traditional server entry point

| File | Purpose |
|---|---|
| `manage.py` | Your main tool — run the server, create migrations, open the shell |
| `settings.py` | Configure databases, installed apps, time zone, static files |
| `urls.py` | The "reception desk" — routes incoming URLs to the right view |
| `wsgi.py` / `asgi.py` | Used when deploying to a production server |

**Quick test**: Start the development server and visit `http://127.0.0.1:8000/`

```bash
python manage.py runserver
```

You should see Django's welcome rocket page 🚀

---

## Creating a Django App

A **project** is the whole website. An **app** is one feature of it.

**Analogy**: If the project is the entire store, the `sales` app handles the till, the `inventory` app tracks stock, and the `staff` app manages employees. Each is its own app inside the same project.

Let's create our `sales` app:

In [ ]:
# Make sure you're inside the posdb/ project folder
python manage.py startapp sales

sales/
    migrations/       ← auto-generated database change history
    __init__.py
    admin.py          ← register models to appear in the admin panel
    apps.py           ← app configuration
    models.py         ← define your database tables as Python classes
    tests.py          ← write automated tests here
    views.py          ← handle requests and return responses
    urls.py           ← (you create this) URL patterns for this app
    templates/        ← (you create this) HTML files

### Register the App

Open `posdb/settings.py` and add `'sales'` to `INSTALLED_APPS`:

In [ ]:
# posdb/settings.py

INSTALLED_APPS = [
    'django.contrib.admin',
    'django.contrib.auth',
    'django.contrib.contenttypes',
    'django.contrib.sessions',
    'django.contrib.messages',
    'django.contrib.staticfiles',
    'sales',   # ← add your app here
]

---

## Models and the Django ORM

A **Model** is a Python class that represents a database table. Each attribute is a column.

Django's **ORM (Object-Relational Mapper)** lets you:
- Create, read, update, and delete records using Python — no raw SQL needed
- Automatically generate database migrations when you change a model

### Our POS Data Model

A typical POS system has three core tables:

```
Product ──< OrderItem >── Order
```

- **Product**: items the store sells (name, price, stock)
- **Order**: a single sale transaction (cashier, timestamp, total)
- **OrderItem**: one line on the receipt — which product, how many, at what price

### Defining the Models

Open `sales/models.py` and define all three models:

In [ ]:
# sales/models.py

from django.db import models


class Product(models.Model):
    CATEGORY_CHOICES = [
        ('food',        'Food & Drink'),
        ('electronics', 'Electronics'),
        ('clothing',    'Clothing'),
        ('household',   'Household'),
        ('other',       'Other'),
    ]

    name       = models.CharField(max_length=200)
    category   = models.CharField(max_length=20, choices=CATEGORY_CHOICES)
    price      = models.DecimalField(max_digits=8, decimal_places=2)   # e.g. 12.99
    stock      = models.PositiveIntegerField(default=0)                # units in stock
    barcode    = models.CharField(max_length=50, unique=True, blank=True)
    is_active  = models.BooleanField(default=True)                     # hide discontinued items

    def __str__(self):
        return f"{self.name}  —  ${self.price}  (stock: {self.stock})"

    class Meta:
        ordering = ['name']


class Order(models.Model):
    STATUS_CHOICES = [
        ('open',       'Open'),
        ('paid',       'Paid'),
        ('refunded',   'Refunded'),
        ('cancelled',  'Cancelled'),
    ]

    cashier    = models.CharField(max_length=100)           # name or employee ID
    status     = models.CharField(max_length=20, choices=STATUS_CHOICES, default='open')
    created_at = models.DateTimeField(auto_now_add=True)    # set when the order is created
    notes      = models.TextField(blank=True)

    @property
    def total(self):
        """Sum the subtotal of every line item on this order."""
        return sum(item.subtotal for item in self.items.all())

    def __str__(self):
        return f"Order #{self.pk}  [{self.status.upper()}]  —  ${self.total:.2f}"

    class Meta:
        ordering = ['-created_at']


class OrderItem(models.Model):
    order     = models.ForeignKey(Order, on_delete=models.CASCADE, related_name='items')
    product   = models.ForeignKey(Product, on_delete=models.PROTECT)   # PROTECT prevents deleting a product that has sales
    quantity  = models.PositiveIntegerField(default=1)
    unit_price = models.DecimalField(max_digits=8, decimal_places=2)   # price at time of sale

    @property
    def subtotal(self):
        return self.unit_price * self.quantity

    def __str__(self):
        return f"{self.quantity} × {self.product.name}  @  ${self.unit_price}"

### Key ORM Concepts Used Above

| Concept | Example | What it means |
|---|---|---|
| `ForeignKey` | `order = ForeignKey(Order, ...)` | Many OrderItems belong to one Order |
| `on_delete=CASCADE` | Order deleted → its OrderItems deleted too |
| `on_delete=PROTECT` | Can't delete a Product that appears on an order |
| `related_name='items'` | `order.items.all()` — access items from the Order side |
| `@property` | `order.total` — computed value, not stored in the DB |

### Running Migrations

In [ ]:
# Step 1: Create migration files
python manage.py makemigrations sales

# Step 2: Apply them to the database
python manage.py migrate

# Expected output:
#   Applying sales.0001_initial... OK

---

## Views and Templates

### Creating Views

A **view** is a Python function that:
1. Receives an HTTP request
2. Queries the database (via the Model)
3. Passes data to a template
4. Returns an HTTP response (the rendered HTML)

Open `sales/views.py` and write three views:

In [ ]:
# sales/views.py

from django.shortcuts import render, get_object_or_404
from .models import Product, Order


def product_list(request):
    """Show all active products, sorted A–Z."""
    products = Product.objects.filter(is_active=True)
    return render(request, 'sales/product_list.html', {'products': products})


def product_detail(request, pk):
    """Show details for a single product. Return 404 if not found."""
    product = get_object_or_404(Product, pk=pk)
    return render(request, 'sales/product_detail.html', {'product': product})


def order_list(request):
    """Show all orders, most recent first."""
    orders = Order.objects.all()
    return render(request, 'sales/order_list.html', {'orders': orders})

### Creating the Product List Template

Create the folder `sales/templates/sales/` and add `product_list.html`:

> **Why the nested `sales/` folder?** Django looks for templates in `templates/`. Nesting by app name avoids name collisions if two apps both have a `product_list.html`.

In [ ]:
<!-- sales/templates/sales/product_list.html -->
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>🛒 POS — Products</title>
    <style>
        body { font-family: Arial, sans-serif; max-width: 900px; margin: 40px auto; padding: 0 20px; background: #f0f4f8; }
        h1   { color: #1a202c; }
        .product-card { background: white; border-radius: 10px; padding: 16px 20px; margin: 12px 0;
                        box-shadow: 0 2px 6px rgba(0,0,0,0.08); display: flex; justify-content: space-between; align-items: center; }
        .product-card h2 a { text-decoration: none; color: #2b6cb0; font-size: 1.1em; }
        .badge { display: inline-block; padding: 3px 10px; border-radius: 12px; font-size: 0.8em;
                 background: #ebf8ff; color: #2b6cb0; }
        .price  { font-size: 1.3em; font-weight: bold; color: #276749; }
        .stock  { font-size: 0.9em; color: #718096; }
        .low    { color: #c53030; font-weight: bold; }
        .empty  { color: #888; font-style: italic; }
        nav a   { margin-right: 16px; color: #2b6cb0; }
    </style>
</head>
<body>
    <h1>🛒 Product Catalogue</h1>
    <nav>
        <a href="/sales/products/">Products</a>
        <a href="/sales/orders/">Orders</a>
        <a href="/admin/">Admin</a>
    </nav>
    <p>{{ products|length }} active product(s) in the system</p>

    {% for product in products %}
    <div class="product-card">
        <div>
            <h2><a href="/sales/products/{{ product.pk }}/">{{ product.name }}</a></h2>
            <span class="badge">{{ product.get_category_display }}</span>
            {% if product.barcode %}
            <span class="stock">Barcode: {{ product.barcode }}</span>
            {% endif %}
        </div>
        <div style="text-align:right">
            <p class="price">${{ product.price }}</p>
            {% if product.stock <= 5 %}
            <p class="low">⚠️ Low stock: {{ product.stock }} left</p>
            {% else %}
            <p class="stock">In stock: {{ product.stock }}</p>
            {% endif %}
        </div>
    </div>
    {% empty %}
    <p class="empty">No products yet — add some via the <a href="/admin/">admin panel</a>!</p>
    {% endfor %}
</body>
</html>

### Creating the Order List Template

Add `order_list.html` to the same folder:

In [ ]:
<!-- sales/templates/sales/order_list.html -->
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>🧾 POS — Orders</title>
    <style>
        body  { font-family: Arial, sans-serif; max-width: 900px; margin: 40px auto; padding: 0 20px; background: #f0f4f8; }
        table { width: 100%; border-collapse: collapse; background: white; border-radius: 10px;
                overflow: hidden; box-shadow: 0 2px 6px rgba(0,0,0,0.08); }
        th    { background: #2b6cb0; color: white; padding: 12px 16px; text-align: left; }
        td    { padding: 11px 16px; border-bottom: 1px solid #e2e8f0; }
        tr:last-child td { border-bottom: none; }
        .paid      { color: #276749; font-weight: bold; }
        .open      { color: #c05621; font-weight: bold; }
        .refunded  { color: #6b46c1; }
        .cancelled { color: #718096; text-decoration: line-through; }
        nav a { margin-right: 16px; color: #2b6cb0; }
    </style>
</head>
<body>
    <h1>🧾 Sales Orders</h1>
    <nav>
        <a href="/sales/products/">Products</a>
        <a href="/sales/orders/">Orders</a>
        <a href="/admin/">Admin</a>
    </nav>
    <p>{{ orders|length }} order(s) recorded</p>

    <table>
        <thead>
            <tr><th>#</th><th>Cashier</th><th>Date</th><th>Status</th><th>Total</th><th>Items</th></tr>
        </thead>
        <tbody>
        {% for order in orders %}
            <tr>
                <td>{{ order.pk }}</td>
                <td>{{ order.cashier }}</td>
                <td>{{ order.created_at|date:"d M Y, H:i" }}</td>
                <td><span class="{{ order.status }}">{{ order.get_status_display }}</span></td>
                <td>${{ order.total }}</td>
                <td>{{ order.items.count }}</td>
            </tr>
        {% empty %}
            <tr><td colspan="6" style="text-align:center;color:#888">No orders yet.</td></tr>
        {% endfor %}
        </tbody>
    </table>
</body>
</html>

---

## URL Routing in Django

URL routing connects a URL pattern (like `/sales/products/3/`) to the view function that handles it.

### Step 1 — App URLs (`sales/urls.py`)

Create a new file `sales/urls.py`:

In [ ]:
# sales/urls.py

from django.urls import path
from . import views

urlpatterns = [
    # /sales/products/       → product catalogue
    path('products/', views.product_list, name='product_list'),

    # /sales/products/1/     → detail page for product with id=1
    path('products/<int:pk>/', views.product_detail, name='product_detail'),

    # /sales/orders/         → all orders
    path('orders/', views.order_list, name='order_list'),
]

### Step 2 — Project URLs (`posdb/urls.py`)

Connect the app's URLs into the main project URL file using `include()`:

In [ ]:
# posdb/urls.py

from django.contrib import admin
from django.urls import path, include

urlpatterns = [
    path('admin/', admin.site.urls),          # /admin/  → Django admin panel
    path('sales/', include('sales.urls')),    # /sales/  → delegate to sales/urls.py
]

# URL flow example:
# Request: GET /sales/products/3/
# Matches: path('sales/', ...) → strips 'sales/', passes 'products/3/' to sales/urls.py
# Matches: path('products/<int:pk>/', ...) → calls product_detail(request, pk=3)

---

## Using Django Admin

Django's admin panel is a **free, auto-generated dashboard** for managing your data — perfect for a store manager to add products, view orders, and update stock.

### Step 1 — Register the Models

Open `sales/admin.py` and register all three models:

In [ ]:
# sales/admin.py

from django.contrib import admin
from .models import Product, Order, OrderItem


@admin.register(Product)
class ProductAdmin(admin.ModelAdmin):
    list_display  = ['name', 'category', 'price', 'stock', 'is_active']
    list_filter   = ['category', 'is_active']
    search_fields = ['name', 'barcode']
    ordering      = ['name']


class OrderItemInline(admin.TabularInline):
    """Show order items directly inside the Order edit page."""
    model  = OrderItem
    extra  = 1    # number of empty rows displayed for adding new items
    fields = ['product', 'quantity', 'unit_price']


@admin.register(Order)
class OrderAdmin(admin.ModelAdmin):
    list_display  = ['pk', 'cashier', 'status', 'created_at']
    list_filter   = ['status']
    search_fields = ['cashier', 'notes']
    ordering      = ['-created_at']
    inlines       = [OrderItemInline]    # ← show items inside the order form

### Step 2 — Create a Superuser Account

In [ ]:
# Create an admin account (you'll be prompted for username, email, password)
python manage.py createsuperuser

### Step 3 — Run the Server and Explore

In [ ]:
python manage.py runserver

# Then open your browser and visit:
# http://127.0.0.1:8000/admin/             ← admin panel
# http://127.0.0.1:8000/sales/products/    ← product catalogue
# http://127.0.0.1:8000/sales/orders/      ← orders list

---

## Bonus: Django Shell — Interact with Your Data

The Django shell is an interactive Python console with your entire project loaded. Use it to seed sample data and test ORM queries.

Open it with:
```bash
python manage.py shell
```

In [ ]:
# ── Run these lines inside: python manage.py shell ──

from sales.models import Product, Order, OrderItem

# ── 1. Create Products ──────────────────────────────────────
coffee = Product.objects.create(
    name='Espresso Coffee',
    category='food',
    price=3.50,
    stock=100,
    barcode='FOOD-001',
)
sandwich = Product.objects.create(
    name='Club Sandwich',
    category='food',
    price=7.99,
    stock=20,
    barcode='FOOD-002',
)
headphones = Product.objects.create(
    name='Wireless Headphones',
    category='electronics',
    price=49.99,
    stock=15,
    barcode='ELEC-001',
)
charger = Product.objects.create(
    name='USB-C Charger',
    category='electronics',
    price=19.99,
    stock=4,      # low stock!
    barcode='ELEC-002',
)

print("Products created!")

# ── 2. Create an Order with Items ──────────────────────────
order1 = Order.objects.create(cashier='Alice', status='paid')

OrderItem.objects.create(
    order=order1,
    product=coffee,
    quantity=2,
    unit_price=coffee.price,
)
OrderItem.objects.create(
    order=order1,
    product=sandwich,
    quantity=1,
    unit_price=sandwich.price,
)

print(f"Order total: ${order1.total:.2f}")   # Should print $14.99

# ── 3. Query Examples ──────────────────────────────────────

# All products, sorted A-Z
all_products = Product.objects.all()
print(f"Total products: {all_products.count()}")

# Only food items
food_items = Product.objects.filter(category='food')
for p in food_items:
    print(p)

# Low stock warning: products with 5 or fewer units
low_stock = Product.objects.filter(stock__lte=5)
print("\n⚠️  Low stock items:")
for p in low_stock:
    print(f"  {p.name}: {p.stock} left")

# Products priced over $10
expensive = Product.objects.filter(price__gt=10).order_by('-price')
for p in expensive:
    print(f"{p.name}: ${p.price}")

# ── 4. Update Stock ────────────────────────────────────────
charger = Product.objects.get(barcode='ELEC-002')
charger.stock += 50    # restock delivery arrived
charger.save()
print(f"Updated stock for {charger.name}: {charger.stock}")

# ── 5. All items on Order #1 ───────────────────────────────
for item in order1.items.all():
    print(f"  {item}  →  subtotal: ${item.subtotal:.2f}")

**ORM Query Cheatsheet**

| Goal | ORM Query |
|---|---|
| All records | `Product.objects.all()` |
| Filter by field | `Product.objects.filter(category='food')` |
| Exclude records | `Product.objects.exclude(is_active=False)` |
| Less than or equal | `Product.objects.filter(stock__lte=5)` |
| Greater than | `Product.objects.filter(price__gt=10)` |
| Order results | `.order_by('-price')` (prefix `-` = descending) |
| Get one record | `Product.objects.get(barcode='FOOD-001')` |
| Get or 404 | `get_object_or_404(Product, pk=1)` |
| Count | `Order.objects.count()` |
| Follow FK reverse | `order.items.all()` — all OrderItems for an order |
| Follow FK forward | `item.product.name` — the product on an OrderItem |

---

## Conclusion

Congratulations! You've built the foundation of a **Point of Sale System** with Django.

### What You Learned

| Concept | What it does |
|---|---|
| **Project vs App** | Project = whole system; App = one feature (e.g. sales) |
| **Model** | Python class → database table (Product, Order, OrderItem) |
| **ForeignKey** | Links models together — an OrderItem belongs to an Order |
| **Migrations** | Auto-generated scripts that update the database schema |
| **View** | Python function that fetches data and renders a template |
| **Template** | HTML file with `{{ }}` and `{% %}` for dynamic content |
| **URL Routing** | Maps URL patterns to view functions |
| **Admin Panel** | Free built-in dashboard; inline editing of related records |
| **ORM Shell** | Interactive way to query the database with Python |

### Your POS App Structure

```
posdb/
├── manage.py
├── posdb/
│   ├── settings.py      ← registered 'sales' app
│   └── urls.py          ← routes /sales/ to sales/urls.py
└── sales/
    ├── models.py        ← Product, Order, OrderItem
    ├── views.py         ← product_list(), product_detail(), order_list()
    ├── urls.py          ← /sales/products/ and /sales/orders/ patterns
    ├── admin.py         ← ProductAdmin, OrderAdmin with inline items
    └── templates/
        └── sales/
            ├── product_list.html
            └── order_list.html
```

### Coming Up Next Week

- **Forms** — Let cashiers create new orders through a web form
- **User Authentication** — Staff login/logout so orders are tied to a user account
- **Deployment** — Put your POS system live on the internet

> **Challenge**: Add a `Discount` model that can be applied to an order, and update `Order.total` to subtract the discount amount!

---

## Homework 🏠

Complete the following exercises to reinforce what you learned today. Each task builds on the POS system you set up in this lesson.

---

### Task 1 — Add More Products (Easy)

Using the Django shell, add **at least 4 more products** across at least **2 different categories**. Make sure each product has a unique barcode.

```python
from sales.models import Product

Product.objects.create(
    name='...',
    category='...',   # food | electronics | clothing | household | other
    price=...,
    stock=...,
    barcode='...',
)
```

---

### Task 2 — ORM Practice (Easy–Medium)

Open the Django shell and run the following queries. Write your results in the answer cell at the bottom.

1. Count the total number of products in the database.
2. List all products in the `electronics` category, ordered cheapest first.
3. Find all products where `stock` is less than or equal to 10 (low stock alert).
4. List all `paid` orders.
5. Get all `OrderItem` records for order `#1` and print each item's subtotal.

---

### Task 3 — Product Detail Template (Medium)

The `product_detail` view already exists in `views.py`, but there is no template for it yet.

Create `sales/templates/sales/product_detail.html` that displays:
- Product name and category badge
- Price
- Barcode
- Stock level (show a warning if stock < 10)
- A "← Back to products" link that goes to `/sales/products/`

---

### Task 4 — Add a New Order via Admin (Medium)

1. Start the server: `python manage.py runserver`
2. Log in at `http://127.0.0.1:8000/admin/`
3. Create a **new Order** with status `paid` and at least **2 line items** using the inline form.
4. Visit `http://127.0.0.1:8000/sales/orders/` and verify the new order and its total appear in the list.

---

### Task 5 — Deactivate a Product (Medium)

In the Django shell:
1. Set `is_active = False` on one product and save it.
2. Confirm it no longer appears at `/sales/products/` (because the view filters by `is_active=True`).
3. Write the two lines of code you used.

---

### Task 6 — Bonus Challenge ⭐

Add a `Discount` model that can be applied to an order:

```python
class Discount(models.Model):
    order       = models.OneToOneField(Order, on_delete=models.CASCADE, related_name='discount')
    description = models.CharField(max_length=200)   # e.g. "Staff discount"
    amount      = models.DecimalField(max_digits=6, decimal_places=2)

    def __str__(self):
        return f"{self.description} (—${self.amount}) on Order #{self.order.pk}"
```

Then:
1. Run migrations for the new model.
2. Register `Discount` in `admin.py`.
3. Update `Order.total` so it subtracts the discount amount if one exists:
   ```python
   @property
   def total(self):
       subtotal = sum(item.subtotal for item in self.items.all())
       try:
           return subtotal - self.discount.amount
       except Discount.DoesNotExist:
           return subtotal
   ```
4. Test it in the shell by creating a `Discount` for an existing order and verifying `order.total` changes.

---

### Answer Space

**Task 2 Results:**

1. Total product count:
2. Electronics (cheapest first):
3. Low stock (≤ 10 units):
4. Paid orders:
5. Order #1 subtotals:

**Task 5 Code:**
```python
# write your two lines here
```

**Notes:**